In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd

from matplotlib import pyplot as plt

In [ ]:
import sys
sys.path.append('/home/zanardi/Codes/TC/HyperNet/hypernet')
sys.path.append('/home/zanardi/Codes/ML/PyCOMET/pycomet')

from hypernet.src.general import utils as hyp_utils
thermophysics = hyp_utils.get_thermophysics_mod()
mix_mod = thermophysics.reaction_thermo.mixture
sp_th_mod = thermophysics.specie_thermo.specie_thermo

import matplotlib.pyplot as plt

In [ ]:
path = '/home/zanardi/Workspace/ML/PyCOMET/chemreac/mt/O3/run.01.T/'
model_path = path + '/saving/mt_O3_mmopnn/SavedModel/'
data_path = path + '/data/test/case.010.csv'

In [ ]:
specie_thermo = {
    0: {
        'specie': {
            'name': 'O2',
            'model': {
               'levels': 'O2g.csv'
            }
        },
        'thermo': {
            'model': 'RigidRotorHarmonicOscillator',
            'Tr_index': 0,
            'Tv_index': 1
        }
    },
    1: {
        'specie': {
            'name': 'O',
            'model': {
               'levels': 'Og.csv'
            }
        },
        'thermo': {
            'model': 'Constant'
        }
    }
}

process = ['adiabatic', 'isochoric']

In [ ]:
from pycomet.networks.chemreacnn import TwoTempThermoLay

x_inp = [tf.keras.Input(shape=(4,)), tf.keras.Input(shape=(1,))]
y_inp = [tf.keras.Input(shape=(2,)), tf.keras.Input(shape=(2,))]
outputs = TwoTempThermoLay(
    process=process,
    specie_thermo=specie_thermo,
    out_indices={'Y': 0, 'T': 1},
    inp_indices={'e': {0: 3}}
)(x_inp, y_inp)
model = tf.keras.Model(inputs=[x_inp, y_inp], outputs=outputs)

In [ ]:
# Specie thermodynamics
sp_th = {}
for index, s in specie_thermo.items():
    args = s if 'config' in s \
        else {**s, **dict(index=index, process=process)}
    sp_th_ = sp_th_mod.SpecieThermo(**args)
    sp_th[sp_th_.specie.name] = sp_th_
for name, s in sp_th.items():
    if s.specie.n_at > 1:
        molecule = name
    else:
        atom = name
# Mixture
mix = mix_mod.MultiComponent(sp_th)

In [ ]:
def get_norm_energies(Y, T):
    # Update thermodynamics
    for s in sp_th.values():
        s.update(T)
    # Update mixture
    mix.update(Y, var='Y')
    # Total energy
    e = mix._he()
    # Formation energy
    ef = 0.
    for s in sp_th.values():
        ef = ef + s.thermo.e_f*s.specie.Y
    # Vibrational energy
    s = sp_th['O2']
    ev = s.thermo.th_int.ho.e*s.specie.Y
    # Roto-translational energy
    ert = 0.
    for k, s in sp_th.items():
        ert = ert + s.thermo.e_tr*s.specie.Y
        if k == 'O2':
            ert = ert + s.thermo.th_int.rr.e*s.specie.Y
    # Normalized energies
    return np.hstack([ert, ev])/(e-ef)

In [ ]:
def get_temperatures(e, e_hat, Y):
    # Update thermodynamics
    mix.update(Y, var='Y')
    # Formation energy
    e_f = mix._e_f()
    # Roto-translational & Vibrational energy
    e_rt, e_v = tf.split(e_hat*(e-e_f), 2, axis=1)
    # Vibrational temperature
    e_v = e_v / sp_th[molecule].specie.Y
    Tv = sp_th[molecule].thermo.th_int.ho._T(e_v)
    # Roto-translational temperature
    cv_rt = tf.zeros_like(e)
    for k, s in sp_th.items():
        cv_rt = cv_rt + s.thermo.cv_tr*s.specie.Y
        if k == molecule:
            cv_rt = cv_rt + s.thermo.th_int.rr.cv*s.specie.Y
    Th = e_rt/cv_rt
    # Return outputs
    return np.hstack([Th, Tv])

In [ ]:
cols = {
    'x': [[0,1,2,3],[4]],
    'y': [[5,6],[7,8]]
}
X = pd.read_csv(data_path).to_numpy(dtype=np.float32)
ic, t = [np.take(X, i, axis=1) for i in cols['x']]
Y, T = [np.take(X, i, axis=1) for i in cols['y']]
e = np.take(ic, [3], axis=1)

In [ ]:
e_hat = get_norm_energies(Y, T)
# T_pred = get_temperatures(e, e_hat, Y)
T_pred = model([[ic, t], [Y, e_hat]])[1]

In [ ]:
plt.semilogx(t, T_pred, '-^')
plt.semilogx(t, T, '--')